<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/mnist_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = "cuda" if torch.cuda.is_available() else "cpu"

In [0]:
# hyper parameter
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [0]:
mnist_train = datasets.MNIST(root="MNIST_data/", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="MNIST_data/", train=False, download=True, transform=transforms.ToTensor())

In [0]:
data_loader = torch.utils.data.DataLoader(dataset=mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)

In [0]:
# fully connected layer 3개
# activation function은 ReLU
linear1 = torch.nn.Linear(784, 256, bias=True).to(device)
linear2 = torch.nn.Linear(256, 256, bias=True).to(device)
linear3 = torch.nn.Linear(256, 10, bias=True).to(device)
relu = torch.nn.ReLU()

In [8]:
# 각 layer의 weight 초기화
torch.nn.init.normal_(linear1.weight)
torch.nn.init.normal_(linear2.weight)
torch.nn.init.normal_(linear3.weight)

Parameter containing:
tensor([[ 1.0114e+00,  7.5326e-01,  1.0995e+00,  ..., -9.7366e-01,
          1.1318e+00,  1.4785e-01],
        [-7.4874e-01,  1.0378e+00, -7.5724e-01,  ...,  8.0703e-01,
          1.4159e+00, -1.1979e+00],
        [-5.6018e-01,  5.0069e-02, -1.3022e-03,  ..., -7.3353e-01,
          1.6938e+00,  3.3569e-01],
        ...,
        [-4.7375e-01,  3.3837e-02, -7.1804e-01,  ...,  7.9079e-01,
         -2.7039e-01,  1.0232e+00],
        [ 1.0389e-01,  2.2732e+00,  8.4060e-01,  ..., -9.2641e-03,
          5.9505e-01,  1.9120e+00],
        [ 1.0814e+00, -7.7754e-01, -5.7703e-01,  ...,  6.2222e-01,
         -3.2546e+00,  1.7123e+00]], device='cuda:0', requires_grad=True)

In [0]:
# model
# Sequential을 이용(이건 keras와 유사하네)
model = torch.nn.Sequential(linear1,relu, linear2, relu, linear3).to(device)

In [0]:
# cost function과 optimizer 설정
cost_function = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
for epoch in range(training_epochs):
  avg_cost = 0
  total_batch = len(data_loader) # len(mnist_train) / batch_size
  for X, Y in data_loader:
    X = X.view(-1, 784).to(device)
    Y = Y.to(device)
    
    optimizer.zero_grad()
    hypothesis = model(X)
    cost = cost_function(hypothesis, Y)
    cost.backward()
    optimizer.step()
    
    avg_cost += cost / total_batch
    
  print("Epoch : {}, cost = {:.9f}".format(epoch+1, avg_cost))
  
print("Learning is finished!!")

Epoch : 1, cost = 144.562591553
Epoch : 2, cost = 37.533958435
Epoch : 3, cost = 23.424444199
Epoch : 4, cost = 16.388782501
Epoch : 5, cost = 11.779355049
Epoch : 6, cost = 8.715884209
Epoch : 7, cost = 6.376405239
Epoch : 8, cost = 4.768291950
Epoch : 9, cost = 3.488099575
Epoch : 10, cost = 2.682116747
Epoch : 11, cost = 2.011949301
Epoch : 12, cost = 1.558253169
Epoch : 13, cost = 1.222649097
Epoch : 14, cost = 0.946227252
Epoch : 15, cost = 0.768682778
Learning is finished!!


In [14]:
import random

with torch.no_grad():
  X_test = mnist_test.test_data.view(-1, 28*28).float().to(device)
  Y_test = mnist_test.test_labels.to(device)
  
  prediction = model(X_test)
  correct_prediction = (torch.argmax(prediction, dim=1) == Y_test)
  accuracy = correct_prediction.float().mean()
  print(accuracy)
  print("Accuracy : ", accuracy.item())
  
  # choose one and predict
  choice_idx = random.randint(0, len(X_test) - 1)
  X_choice = X_test[choice_idx:choice_idx+1].view(-1, 784).float().to(device) 
  Y_choice = Y_test[choice_idx:choice_idx+1].to(device)
  
  print(Y_choice)
  print("Label : ", Y_choice.item())
  sample_pred = model(X_choice)
  print("Prediction : ", torch.argmax(sample_pred, 1).item())

tensor(0.9458, device='cuda:0')
Accuracy :  0.9457999467849731
tensor([2], device='cuda:0')
Label :  2
Prediction :  2


/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:58: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:48: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")
